# 🧹 Paso 2 — Limpieza y Preprocesamiento
**Instrucción:** Asegúrate de haber ejecutado primero `01_eda.ipynb`.
Cambia `ARCHIVO` y `COLUMNA_OBJETIVO` según tu dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
print('✅ Librerías cargadas')

## 1. Cargar el dataset original

In [ ]:
# ⚠️ CAMBIA ESTOS DOS VALORES
ARCHIVO = 'tu_dataset.csv'
COLUMNA_OBJETIVO = 'target'

df = pd.read_csv(ARCHIVO)
print(f'Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas')
df.head()

## 2. Manejo de valores nulos

In [ ]:
nulos_antes = df.isnull().sum().sum()
print(f'Valores nulos antes: {nulos_antes}')

# Columnas numéricas: rellenar con la mediana
numericas = df.select_dtypes(include=[np.number]).columns.tolist()
for col in numericas:
    if df[col].isnull().sum() > 0:
        mediana = df[col].median()
        df[col].fillna(mediana, inplace=True)
        print(f'  · {col}: nulos rellenados con mediana ({mediana})')

# Columnas categóricas: rellenar con la moda
categoricas = df.select_dtypes(include=['object']).columns.tolist()
for col in categoricas:
    if df[col].isnull().sum() > 0:
        moda = df[col].mode()[0]
        df[col].fillna(moda, inplace=True)
        print(f'  · {col}: nulos rellenados con moda ({moda})')

nulos_despues = df.isnull().sum().sum()
print(f'Valores nulos después: {nulos_despues}')
print('✅ Manejo de nulos completado')

## 3. Codificación de variables categóricas

In [ ]:
categoricas = df.select_dtypes(include=['object']).columns.tolist()
if COLUMNA_OBJETIVO in categoricas:
    categoricas.remove(COLUMNA_OBJETIVO)

if len(categoricas) == 0:
    print('✅ No hay variables categóricas para codificar')
else:
    le = LabelEncoder()
    for col in categoricas:
        df[col] = le.fit_transform(df[col])
        print(f'  · {col}: codificada con LabelEncoder')
    print('✅ Codificación completada')

df.head()

## 4. Detección y manejo de duplicados

In [ ]:
duplicados = df.duplicated().sum()
print(f'Filas duplicadas encontradas: {duplicados}')
if duplicados > 0:
    df.drop_duplicates(inplace=True)
    print(f'✅ Duplicados eliminados. Filas restantes: {len(df)}')
else:
    print('✅ No hay duplicados')

## 5. Separar features y variable objetivo

In [ ]:
X = df.drop(COLUMNA_OBJETIVO, axis=1)
y = df[COLUMNA_OBJETIVO]

print(f'Features (X): {X.shape}')
print(f'Objetivo  (y): {y.shape}')
print(f'Columnas usadas: {list(X.columns)}')

## 6. Normalización con StandardScaler

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print('Estadísticas ANTES de normalizar:')
print(X.describe().loc[['mean','std']].round(3))
print('\nEstadísticas DESPUÉS de normalizar:')
print(X_scaled.describe().loc[['mean','std']].round(3))

## 7. Comparación visual antes/después de normalizar

In [ ]:
cols_muestra = X.columns[:4].tolist()
fig, axes = plt.subplots(2, len(cols_muestra), figsize=(14, 6))

for i, col in enumerate(cols_muestra):
    sns.histplot(X[col], kde=True, ax=axes[0][i], color='#DD8452')
    axes[0][i].set_title(f'{col}\n(antes)')
    sns.histplot(X_scaled[col], kde=True, ax=axes[1][i], color='#4C72B0')
    axes[1][i].set_title(f'{col}\n(después)')

plt.suptitle('Efecto de la normalización', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/figures/normalizacion.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. División Train / Test (80% / 20%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Entrenamiento: {X_train.shape[0]} filas ({X_train.shape[0]/len(X_scaled)*100:.0f}%)')
print(f'Prueba:        {X_test.shape[0]} filas ({X_test.shape[0]/len(X_scaled)*100:.0f}%)')
print(f'\nDistribución de clases en train:\n{y_train.value_counts()}')
print(f'\nDistribución de clases en test:\n{y_test.value_counts()}')

## 9. Guardar datos procesados

In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)

X_train.to_csv('data/processed/X_train.csv', index=False)
X_test.to_csv('data/processed/X_test.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)

print('✅ Datos guardados en data/processed/')
print('   · X_train.csv')
print('   · X_test.csv')
print('   · y_train.csv')
print('   · y_test.csv')
print('\n✅ Preprocesamiento completado. Continúa con 03_knn.ipynb')